# 📦 Olist E-commerce — Exploratory Data Analysis (Operation Performance)
### From the Gold Mart to a board-room decision

**Author:** HoongJun · **Module 5 (Analytics)** · **Data:** `sctp-team2-project2-elt.olist_gold_mart_prod`

---

## Executive summary — read this first

> **The company is *buying* growth it should be *earning*.** The binding constraint
> on profitable growth is the **repeat-purchase rate**, and the single biggest
> controllable lever on it is **delivery experience**.

Three executives each see one face of the same elephant:

| Who | What they see | What they conclude | Why that's the wrong fix |
| --- | --- | --- | --- |
| **CEO** | GMV grows, but marketing spend climbs to sustain it; margins soft | "We're working harder to grow" | The leak is *retention*, not pricing or competition |
| **CMO** | Re-engagement underperforms, churn ugly | "Our brand isn't sticky → spend more on loyalty" | The churn cause is operational, not a comms problem |
| **CTO** | Delivery complaints, low review scores | "Logistics cost to optimise" (noise) | Those late deliveries *are* the revenue leak |

**What the data shows (validated in this notebook):**

1. **Retention is the constraint** — only **~3.5%** of orders are repeat purchases
   (96,096 customers → 99,441 orders). Growth ≈ pure new-customer acquisition.
2. **Delivery's tail is toxic** — ~**8%** of delivered orders arrive *late*.
3. **The smoking gun** — on-time orders average **4.3★**; late orders average
   **2.6★**, and **54%** of late-delivery customers leave a 1–2★ review.

**The chain:** *late delivery → low review → customer never returns → low repeat
rate → rising acquisition spend & soft margins.* The cause lives in the **CTO's**
domain, surfaces as the **CMO's** retention problem, and shows up in the **CEO's**
P&L. **Fixing delivery reliability is cheaper than buying replacement growth.**

---

## 1. How to read this notebook (and a 60-second pandas/seaborn primer)

This notebook is also a **learning artifact**. Every analysis step has three parts:

1. **🎯 Purpose** — *why* we run it (the business question).
2. **🧮 The code** — *what* it does, with comments on each pandas/seaborn idiom.
3. **🔍 Reading the result** — *how* to interpret the numbers/chart and what it implies.

**Vocabulary you'll see a lot:**

| Term | Plain meaning |
| --- | --- |
| `DataFrame` (`df`) | A table in memory — rows × named columns. Think "Excel sheet in Python." |
| `df.groupby('x')['y'].mean()` | "For each value of column *x*, average column *y*." (a pivot/summary) |
| `df['col']` | One column (a `Series`). `df[['a','b']]` selects several columns. |
| `df[df['x'] > 5]` | **Filtering** — keep only rows where the condition is True. |
| `.value_counts()` | Frequency table — how many rows fall in each category. |
| `sns.barplot(...)` | Seaborn = matplotlib with nicer defaults. `sns.*plot` draws the chart. |

You do **not** need to memorise these — the comments re-explain them in context.

### 1.1 Setup — connect to BigQuery

We read **gold-mart** tables (the trusted, modelled layer) directly into pandas.

**Authentication:** we use a **service-account keyfile** stored in `../secrets/`. To
run elsewhere, point `GOOGLE_APPLICATION_CREDENTIALS` at your own key — or, if no key
is found, the notebook falls back to your local `gcloud` login (ADC) so it still runs.

In [1]:
!pip install -q seaborn db-dtypes


In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery

warnings.filterwarnings("ignore")            # keep the notebook output clean for readers

# --- Visual defaults so every chart looks consistent & presentation-ready ---
sns.set_theme(style="whitegrid", palette="deep")   # whitegrid = light gridlines, easy to read
plt.rcParams["figure.figsize"] = (10, 5)           # default chart size (width, height in inches)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["figure.dpi"] = 110                   # crispness on screen
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")  # 1,234.50 instead of 1234.4999

PROJECT = "sctp-team2-project2-elt"
GOLD    = f"{PROJECT}.olist_gold_mart_prod"   # the gold mart (our contract)
SNAP    = f"{PROJECT}.snapshots"              # reviews_dbt_scd_snapshot lives here (has order_id)

# --- Build a BigQuery client using a service-account keyfile. ---
# Override the location anytime via:  export GOOGLE_APPLICATION_CREDENTIALS=/path/to/key.json
# Default path is relative to this notebook (notebooks/eda/) -> ../secrets/<key>.json
KEYFILE = os.environ.get(
    "GOOGLE_APPLICATION_CREDENTIALS",
    os.path.join("..", "secrets", "sctp-team2-project2-elt-ROTATED-dbcb3cd092f4.json"),
)

def make_client():
    # Authenticate with the service-account keyfile (the project's chosen method)...
    if os.path.exists(KEYFILE):
        try:
            from google.oauth2 import service_account
            creds = service_account.Credentials.from_service_account_file(KEYFILE)
            client = bigquery.Client(credentials=creds, project=PROJECT)
            client.query("SELECT 1").result()          # smoke-test the credentials
            print(f"✅ Authenticated with service-account keyfile: {KEYFILE}")
            return client
        except Exception as e:
            print(f"⚠️  Keyfile present but unusable ({str(e)[:60]}...). Falling back to gcloud ADC.")
    # ...fall back to Application Default Credentials (your `gcloud auth` login) if no key.
    client = bigquery.Client(project=PROJECT)
    client.query("SELECT 1").result()
    print("✅ Authenticated with gcloud Application Default Credentials (ADC).")
    return client

client = make_client()

def q(sql: str) -> pd.DataFrame:
    # Run SQL on BigQuery and return the result as a pandas DataFrame.
    return client.query(sql).to_dataframe()

E0000 00:00:1781146113.914110 63063776 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1781146113.914125 63063776 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1781146113.914127 63063776 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1781146113.914128 63063776 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1781146113.914129 63063776 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


✅ Authenticated with gcloud Application Default Credentials (ADC).


## 2. Data overview — what are we working with?

**🎯 Purpose:** before *any* analysis, understand the **grain** (what one row means),
the **size**, the **time span**, and where data is **missing**. Skipping this is how
analysts produce confident-but-wrong charts.

**The gold mart (star schema):**

```
                 dim_customers ──┐
                 dim_products  ──┤
                 dim_sellers   ──┼──< fact_orders >── reviews_dbt_scd_snapshot
                                 │     (1 row per order ITEM)      (1 row per review)
```

`fact_orders` is at **order-item grain** — an order with 3 items = 3 rows. So to talk
about *orders* we must aggregate items up to the order. We do that next.

In [3]:
# Pull ONE row per ORDER by aggregating the item-grain fact up to order grain.
# SUM(price) = order GMV; COUNT(*) = items in the order. ANY_VALUE picks the order-level
# attributes (identical across an order's items, so "any value" is safe).
sql_orders = f'''
WITH order_items AS (
  SELECT
    id                                   AS order_id,
    ANY_VALUE(customer_id)               AS customer_id,
    ANY_VALUE(order_status)              AS order_status,
    ANY_VALUE(order_purchase_timestamp)  AS purchase_ts,
    ANY_VALUE(order_approved_at)         AS approved_ts,
    ANY_VALUE(order_delivered_customer_date) AS delivered_ts,
    ANY_VALUE(order_estimated_delivery_date) AS estimated_ts,
    ANY_VALUE(payment_value)             AS payment_value,
    SUM(price)                           AS order_gmv,      -- product revenue for the order
    SUM(freight_value)                   AS order_freight,  -- shipping charged
    COUNT(*)                             AS item_count
  FROM `{GOLD}.fact_orders`
  GROUP BY id
),
cur_reviews AS (   -- current review per order (snapshot rows still valid today)
  SELECT order_id, MAX(review_score) AS review_score
  FROM `{SNAP}.reviews_dbt_scd_snapshot`
  WHERE dbt_valid_to IS NULL AND order_id IS NOT NULL
  GROUP BY order_id
)
SELECT
  o.*,
  c.customer_unique_id,           -- the SAME person across multiple orders (key to retention!)
  c.customer_state,
  r.review_score
FROM order_items o
LEFT JOIN `{GOLD}.dim_customers` c ON o.customer_id = c.id
LEFT JOIN cur_reviews            r ON o.order_id    = r.order_id
'''
orders = q(sql_orders)

# Make timestamps real datetimes so we can do date maths (subtract, resample by month).
for col in ["purchase_ts", "approved_ts", "delivered_ts", "estimated_ts"]:
    orders[col] = pd.to_datetime(orders[col])

print(f"orders table: {orders.shape[0]:,} rows  ×  {orders.shape[1]} columns")
orders.head(3)

orders table: 99,441 rows  ×  14 columns


,order_id,customer_id,order_status,purchase_ts,approved_ts,delivered_ts,estimated_ts,payment_value,order_gmv,order_freight,item_count,customer_unique_id,customer_state,review_score
0,2e7a8482f6fb09756ca50c10d7bfc047,08c5351a6aca1c1589a38f244edeee9d,shipped,2016-09-04 21:15:19+00:00,2016-10-07 13:18:03+00:00,NaT,2016-10-20 00:00:00+00:00,136.23,72.89,63.34,2,b7d76e111c89f7ebf14761390f0f7d17,RR,1
1,30ac9eae54e817b40c3c77fdb444a1e0,d2dacbeeaec5aa8722102ff00b782580,delivered,2017-02-20 18:03:43+00:00,2017-02-20 18:15:12+00:00,2017-03-10 08:07:54+00:00,2017-03-31 00:00:00+00:00,124.18,99.00,25.18,1,1ba7e2fe9ad84553996f4eb8d495782f,PI,1
2,2ec030d65d7742d4afb8df93dd86ba76,37c4e0723167867a40efbaa01ce1c69a,unavailable,2017-03-06 10:24:34+00:00,2017-03-06 10:35:25+00:00,NaT,2017-04-06 00:00:00+00:00,105.04,NaN,NaN,1,4a490384e627a8c422a55f59ae589952,SE,1


**🧮 What just happened:** `orders.shape` returns `(rows, columns)`. `.head(3)` shows
the first 3 rows so we can eyeball the structure. One row = one **order**.

In [4]:
# .info() = the column inventory: name, count of NON-null values, and data type (dtype).
# Where "Non-Null Count" < total rows, that column has MISSING values.
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   order_id            99441 non-null  object             
 1   customer_id         99441 non-null  object             
 2   order_status        99441 non-null  object             
 3   purchase_ts         99441 non-null  datetime64[us, UTC]
 4   approved_ts         99281 non-null  datetime64[us, UTC]
 5   delivered_ts        96476 non-null  datetime64[us, UTC]
 6   estimated_ts        99441 non-null  datetime64[us, UTC]
 7   payment_value       99440 non-null  float64            
 8   order_gmv           98666 non-null  float64            
 9   order_freight       98666 non-null  float64            
 10  item_count          99441 non-null  Int64              
 11  customer_unique_id  99441 non-null  object             
 12  customer_state      99441 non-nu